# Differential Accessibility (DESeq2 on consensus peaks) - headless

Reproduces a paper's differential-peak finding from a consensus-peak count matrix.


In [ ]:
# Parameters (injected at launch). All are str/number so injection stays valid R.
counts_path <- "/data/consensus_peaks.featureCounts.txt"   # featureCounts over consensus peaks
output_path <- "/outputs/da_results.csv"
test_samples <- ""                            # comma-separated sample column names (treatment)
reference_samples <- ""                       # comma-separated sample column names (control)
lfc_threshold <- 1.0
padj_threshold <- 0.05


In [ ]:
suppressMessages(library(DESeq2))

test_s <- trimws(strsplit(test_samples, ",")[[1]]); test_s <- test_s[test_s != ""]
ref_s  <- trimws(strsplit(reference_samples, ",")[[1]]); ref_s <- ref_s[ref_s != ""]
stopifnot(length(test_s) > 0, length(ref_s) > 0)
samples <- c(test_s, ref_s)
condition <- factor(c(rep("test", length(test_s)), rep("reference", length(ref_s))), levels = c("reference", "test"))
coldata <- data.frame(condition = condition, row.names = samples)

# featureCounts writes a leading '# Program...' comment line; comment.char skips it.
mat <- read.delim(counts_path, check.names = FALSE, comment.char = '#', stringsAsFactors = FALSE)
coord <- c("Chr", "Start", "End")
if (!all(coord %in% colnames(mat))) stop("expected featureCounts Chr/Start/End columns")
missing <- setdiff(samples, colnames(mat))
if (length(missing) > 0) stop(paste("samples not in matrix:", paste(missing, collapse=", ")))
counts <- as.matrix(mat[, samples, drop = FALSE])
counts <- matrix(as.integer(round(as.numeric(counts))), nrow = nrow(counts), dimnames = dimnames(counts))
rownames(counts) <- make.unique(as.character(mat$Geneid))

dds <- DESeqDataSetFromMatrix(countData = counts, colData = coldata, design = ~ condition)
dds <- DESeq(dds)
res <- as.data.frame(results(dds, contrast = c("condition", "test", "reference")))
dir.create(dirname(output_path), showWarnings = FALSE, recursive = TRUE)
out <- data.frame(chr = mat$Chr, start = mat$Start, end = mat$End,
                  log2FoldChange = res$log2FoldChange, padj = res$padj)
write.csv(out, output_path, row.names = FALSE)
cat("wrote", nrow(out), "peaks to", output_path, "\n")
